# Jinja Customer Support Intent Detection

To run this notebook, create a virtual environment and install the packages provided in the `requirements.txt`.

### Step 1: Import the necessary packages

In [1]:
from pathlib import Path
import json
from collections import Counter
from jinja2 import Environment, FileSystemLoader, StrictUndefined

### Step 2: Create the project structure 

In [2]:
DATA_DIR = Path("data")
TEMPLATES_DIR = Path("templates")
OUTPUTS_DIR = Path("outputs")

for directory in [DATA_DIR, TEMPLATES_DIR, OUTPUTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


### Step3: Create a small dataset of support requests

In [3]:
support_requests = [
    {
        "id": "req001",
        "customer_name": "Max Mustermann",
        "text": "The application crashes every time I try to export CSV files larger than 10MB. I've attached the error log.",
        "product_type": "data_analytics_tool",
        "categories": ["export", "crash", "performance"],
        "urgency_level": "high",
        "contains_sarcasm": False,
        "language": "en",
        "expected_intent": "bug_report",
        "has_attachments": True,
        "os": "Windows 11",
        "additional_info": "This started after the last update."
    },
    {
        "id": "req002",
        "customer_name": "Rainer Zufall",
        "text": "Oh fantastic, the 'Improve User Experience' update broke the entire user experience. Now nothing works.",
        "product_type": "mobile_app",
        "categories": ["ui", "update", "bug"],
        "urgency_level": "critical",
        "contains_sarcasm": True,
        "language": "en",
        "expected_intent": "bug_report",
        "has_attachments": False,
        "os": "iOS 17",
        "additional_info": None
    },
    {
        "id": "req003",
        "customer_name": "Ernst Haft",
        "text": "I need help with API authentication and also with setting up webhook notifications for our enterprise plan.",
        "product_type": "api_service",
        "categories": ["api", "authentication", "webhooks", "enterprise"],
        "urgency_level": "medium",
        "contains_sarcasm": False,
        "language": "en",
        "expected_intent": "technical_support",
        "has_attachments": False,
        "os": None,
        "additional_info": "We have a team of 5 developers waiting on this."
    },
    {
        "id": "req004",
        "customer_name": "Anna Nass",
        "text": "The database connection times out after exactly 30 minutes of inactivity. Is there a configurable timeout setting?",
        "product_type": "backend_service",
        "categories": ["database", "timeout", "configuration"],
        "urgency_level": "medium",
        "contains_sarcasm": False,
        "language": "en",
        "expected_intent": "technical_question",
        "has_attachments": False,
        "os": "Linux (Ubuntu 22.04)",
        "additional_info": None
    },
    {
        "id": "req005",
        "customer_name": "Marga Milch",
        "text": "Your documentation says the feature is available, but I can't find it anywhere in the dashboard. Great job on the hide and seek implementation.",
        "product_type": "saas_platform",
        "categories": ["documentation", "ui", "feature_access"],
        "urgency_level": "low",
        "contains_sarcasm": True,
        "language": "en",
        "expected_intent": "feature_request",
        "has_attachments": False,
        "os": None,
        "additional_info": None
    }
]

(DATA_DIR / "support_requests.json").write_text(json.dumps(support_requests, indent=2), encoding="utf-8")
print(f"Wrote {len(support_requests)} support requests.")

Wrote 5 support requests.


### Step 4: Create few-shot examples for intent detection

In [4]:
examples = [
    {
        "text": "The app crashes when I try to export data.",
        "intent": "bug_report",
        "reason": "User reports a malfunction."
    },
    {
        "text": "How do I reset my password?",
        "intent": "technical_question",
        "reason": "User asks for help with a feature."
    },
    {
        "text": "I need to add more users to my account.",
        "intent": "feature_request",
        "reason": "User wants additional functionality."
    },
    {
        "text": "Great, another bug in your update.",
        "intent": "bug_report",
        "reason": "Sarcastic remark about a problem."
    },
    {
        "text": "Why am I being charged for unused features?",
        "intent": "billing",
        "reason": "User questions charges."
    }
]

(DATA_DIR / "few_shot_examples.json").write_text(json.dumps(examples, indent=2), encoding="utf-8")
print(f"Wrote {len(examples)} few-shot examples.")

Wrote 5 few-shot examples.


### Step 5: Create task configurations

In [5]:
task_configs = {
    "project": {
        "name": "Customer Support Intent Detection",
        "description": "Jinja templates for dynamic intent detection prompts in customer support.",
        "default_language": "English"
    },
    "tasks": [
        {
            "name": "intent_classification",
            "mode": "document",
            "labels": ["bug_report", "technical_question", "feature_request", "billing", "feedback"],
            "allow_mixed": False,
            "include_few_shot": True,
            "include_reasoning_field": True,
            "json_only": True,
            "max_examples": 3
        },
        {
            "name": "intent_with_urgency",
            "mode": "document",
            "labels": ["bug_report", "technical_question", "feature_request", "billing", "feedback"],
            "allow_mixed": False,
            "include_few_shot": True,
            "include_reasoning_field": True,
            "json_only": True,
            "max_examples": 5
        },
        {
            "name": "category_based",
            "mode": "category",
            "labels": ["bug_report", "technical_question", "feature_request", "billing", "feedback", "not_applicable"],
            "allow_mixed": False,
            "include_few_shot": False,
            "include_reasoning_field": False,
            "json_only": True,
            "max_examples": 0
        }
    ]
}

(DATA_DIR / "task_configs.json").write_text(json.dumps(task_configs, indent=2), encoding="utf-8")
print(f"Wrote {len(task_configs['tasks'])} task configurations.")

Wrote 3 task configurations.


### Step 6: Define Jinja Templates

The `templates` directory contains three jinja templates (and one macro template) that are based on the ones defined in the sentiment demo discussed during the exercise:

- `intent_prompt.j2`: Single support request containing logic to adjust the prompt based on the mode, included sarcasm, ...
- `batch_prompt.j2`: Multiple support requests using the for-loop logic
- `report.md.j2`: For summirizing the dataset and project in a Markdown report. This also contains the explanation why Jinja is useful here
- (`macros.j2`: Macros for list formatting)

### Step 7: Render the request intent detection prompts

In [6]:
# Define the environment
env = Environment(
    loader=FileSystemLoader(TEMPLATES_DIR),
    undefined=StrictUndefined,
    trim_blocks=True,
    lstrip_blocks=True,
)

# Get the temlates
single_prompt = env.get_template("intent_prompt.txt.j2")
batch_prompt = env.get_template("batch_prompt.txt.j2")
report = env.get_template("report.md.j2")

# Get the project and tasks
project = task_configs["project"]
tasks = task_configs["tasks"]

# Loop over the tasks
for task in tasks:

    # Define the output dir
    task_output_dir = OUTPUTS_DIR / task["name"]
    task_output_dir.mkdir(parents=True, exist_ok=True)

    # Render all the requests using the single request template
    for support_req in support_requests:
        rendered = single_prompt.render(
            project=project,
            task=task,
            request=support_req,
            examples=examples,
        )

        # Add the rendered template to the output dir
        (task_output_dir / f"{support_req['id']}.txt").write_text(rendered, encoding="utf-8")

    # Render all the support requests using the batch_prompt template and save the result 
    rendered_batch = batch_prompt.render(task=task, requests=support_requests)
    (task_output_dir / "batch_prompt.txt").write_text(rendered_batch, encoding="utf-8")

# Get the ground truth
gold_distribution = dict(Counter(support_req["expected_intent"] for support_req in support_requests))

# Render the report and save it as well
rendered_report = report.render(
    project=project,
    requests=support_requests,
    tasks=tasks,
    gold_distribution=gold_distribution,
)
(OUTPUTS_DIR / "project_report.md").write_text(rendered_report, encoding="utf-8")


# List the generated files
print("Generated files:")
for path in sorted(OUTPUTS_DIR.rglob("*")):
    if path.is_file():
        print("-", path)


Generated files:
- outputs/category_based/batch_prompt.txt
- outputs/category_based/req001.txt
- outputs/category_based/req002.txt
- outputs/category_based/req003.txt
- outputs/category_based/req004.txt
- outputs/category_based/req005.txt
- outputs/intent_classification/batch_prompt.txt
- outputs/intent_classification/req001.txt
- outputs/intent_classification/req002.txt
- outputs/intent_classification/req003.txt
- outputs/intent_classification/req004.txt
- outputs/intent_classification/req005.txt
- outputs/intent_with_urgency/batch_prompt.txt
- outputs/intent_with_urgency/req001.txt
- outputs/intent_with_urgency/req002.txt
- outputs/intent_with_urgency/req003.txt
- outputs/intent_with_urgency/req004.txt
- outputs/intent_with_urgency/req005.txt
- outputs/project_report.md
